In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [22]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datetime import datetime

In [3]:
# Canonical paths
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
checkpoints_dir = f"{base_dir}/checkpoints"
splits_dir = f"{preprocessed_dir}/splits"
split_features_dir = f"{preprocessed_dir}/split_features"

results_dir = f"{base_dir}/eval_results"
explain_dir = os.path.join(results_dir, "explainability")
os.makedirs(explain_dir, exist_ok=True)

In [5]:
print("--- Paths resolved: ---")
print(" base_dir:", base_dir)
print(" preprocessed_dir:", preprocessed_dir)
print(" checkpoints_dir:", checkpoints_dir)
print(" results_dir:", results_dir)
print(" explain_dir:", explain_dir)

--- Paths resolved: ---
 base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
 preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
 checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints
 results_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results
 explain_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/explainability


In [6]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Determinism flags (safe defaults)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [9]:
# Load configs + checkpoint + predictions
fusion_config_path = os.path.join(checkpoints_dir, "fusion_config.json")
meta_config_path = os.path.join(checkpoints_dir, "meta_config.json")
model_ckpt_path = os.path.join(checkpoints_dir, "coldfusionx_model_epoch3.pt")

pred_path = os.path.join(results_dir, "test_predictions.npz")

In [10]:
# Load JSON configs
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

# Load predictions (Notebook 09 output)
pred_data = np.load(pred_path)
pred_keys = list(pred_data.keys())

In [11]:
print("--- Loaded Artifacts ---")
print("fusion_config_path:", fusion_config_path)
print("meta_config_path:", meta_config_path)
print("model_ckpt_path:", model_ckpt_path)
print("pred_path:", pred_path)
print("pred_keys:", pred_keys)

# Print key values (audit-friendly)
print("\n--- Key Config Values ---")
print("d_model:", fusion_config.get("d_model"))
print("hidden_dim:", fusion_config.get("hidden_dim"))
print("num_experts:", fusion_config.get("num_experts"))
print("num_scenes:", fusion_config.get("num_scenes"))
print("FEWSHOT_N:", meta_config.get("FEWSHOT_N"))

--- Loaded Artifacts ---
fusion_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/fusion_config.json
meta_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/meta_config.json
model_ckpt_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/coldfusionx_model_epoch3.pt
pred_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/test_predictions.npz
pred_keys: ['y_true', 'y_logit', 'y_prob', 'cold_mask', 'scene_idx', 'alpha', 'gates']

--- Key Config Values ---
d_model: 256
hidden_dim: 256
num_experts: 4
num_scenes: 5
FEWSHOT_N: 10


In [13]:
# Quick sanity on predictions shapes
print("--- Prediction Array Shapes ---")
for k in ["y_true", "y_logit", "y_prob", "cold_mask", "scene_idx", "alpha"]:
    if k in pred_data:
        arr = pred_data[k]
        print(f"{k:10s}: {arr.shape} | dtype={arr.dtype}")

--- Prediction Array Shapes ---
y_true    : (373033,) | dtype=float32
y_logit   : (373033,) | dtype=float32
y_prob    : (373033,) | dtype=float32
cold_mask : (373033,) | dtype=bool
scene_idx : (373033,) | dtype=int64
alpha     : (373033, 1) | dtype=float32


 #### Load Predictions + Build a “Selection Table”

In [23]:
# Load arrays from test_predictions.npz
y_true = pred_data["y_true"].astype(np.float32)
y_logit = pred_data["y_logit"].astype(np.float32)
y_prob = pred_data["y_prob"].astype(np.float32)
cold_mask = pred_data["cold_mask"].astype(bool)
scene_idx = pred_data["scene_idx"].astype(np.int64)
alpha = pred_data["alpha"].astype(np.float32).squeeze()   # (N,)

In [24]:
# gates optional
gates = pred_data["gates"].astype(np.float32) if "gates" in pred_data else None  # (N, E)

In [25]:
N = len(y_true)
assert alpha.shape == (N,), f"alpha shape mismatch: {alpha.shape} vs {(N,)}"

In [26]:
# row_idx mapping
test_idx_path = os.path.join(splits_dir, "test_idx.npy")
test_idx = np.load(test_idx_path).astype(np.int64)  # row indices into row-aligned feature stores
assert test_idx.shape[0] == N, f"test_idx mismatch: {test_idx.shape[0]} vs N={N}"

In [28]:
# Basic DataFrame
pred_df = pd.DataFrame({
    "row_idx": test_idx,
    "y_true": y_true,
    "y_logit": y_logit,
    "y_prob": y_prob,
    "cold_mask": cold_mask,
    "scene_idx": scene_idx,
    "alpha": alpha,
})

print("pred_df created:", pred_df.shape)
pred_df.head()

pred_df created: (373033, 7)


,row_idx,y_true,y_logit,y_prob,cold_mask,scene_idx,alpha
0,2296852,0.0,0.321191,0.579614,False,2,1.0
1,495625,1.0,2.395261,0.916465,False,2,1.0
2,2437289,0.0,-2.105106,0.108602,False,3,1.0
3,1517928,0.0,-5.083009,0.006163,False,2,1.0
4,542839,1.0,0.955819,0.722284,False,2,1.0


Add derived columns (TP/FP/TN/FN, confidence, alpha bins, gates diagnostics)

In [33]:
# Derived columns
THRESH = 0.5  # keeping it fixed
y_pred = (pred_df["y_prob"].values >= THRESH).astype(np.int32)
y_true_int = pred_df["y_true"].values.astype(np.int32)

pred_df["y_pred"] = y_pred
pred_df["confidence"] = np.abs(pred_df["y_prob"].values - 0.5).astype(np.float32)

In [34]:
pred_df["is_tp"] = (y_pred == 1) & (y_true_int == 1)
pred_df["is_fp"] = (y_pred == 1) & (y_true_int == 0)
pred_df["is_tn"] = (y_pred == 0) & (y_true_int == 0)
pred_df["is_fn"] = (y_pred == 0) & (y_true_int == 1)

In [36]:
# Expert chosen (argmax) and max weight
pred_df["gate_expert"] = gates.argmax(axis=1).astype(np.int64)
pred_df["gate_max"] = gates.max(axis=1).astype(np.float32)

# Gate entropy (how "confident" the router is)
eps = 1e-12
p = np.clip(gates, eps, 1.0)
entropy = -(p * np.log(p)).sum(axis=1)  # natural log
pred_df["gate_entropy"] = entropy.astype(np.float32)

In [37]:
# Alpha bins (aligns with "meta-driven vs ID-driven" worked on notebook 09)
pred_df["alpha_bin"] = pd.cut(
    pred_df["alpha"],
    bins=[-np.inf, 0.33, 0.66, np.inf],
    labels=["alpha_low", "alpha_mid", "alpha_high"]
).astype(str)

In [38]:
print("Derived columns added.")
pred_df.head()

Derived columns added.


,row_idx,y_true,y_logit,y_prob,cold_mask,scene_idx,alpha,y_pred,confidence,is_tp,is_fp,is_tn,is_fn,alpha_bin,gate_expert,gate_max,gate_entropy
0,2296852,0.0,0.321191,0.579614,False,2,1.0,1,0.079614,False,True,False,False,alpha_high,0,0.316574,1.350762
1,495625,1.0,2.395261,0.916465,False,2,1.0,1,0.416465,True,False,False,False,alpha_high,0,0.316574,1.350762
2,2437289,0.0,-2.105106,0.108602,False,3,1.0,0,0.391398,False,False,True,False,alpha_high,2,0.650946,1.026001
3,1517928,0.0,-5.083009,0.006163,False,2,1.0,0,0.493837,False,False,True,False,alpha_high,0,0.316574,1.350762
4,542839,1.0,0.955819,0.722284,False,2,1.0,1,0.222284,True,False,False,False,alpha_high,0,0.316574,1.350762


#### Selection Table Summary

In [55]:
print("N total:", len(pred_df))
print("Cold rate:", float(pred_df["cold_mask"].mean()))
print("Positive rate:", float(pred_df["y_true"].mean()))

N total: 373033
Cold rate: 0.038975104079263766
Positive rate: 0.3367503583431244


In [56]:
print("--- Counts by cold/warm: ---\n")
print(pred_df.groupby("cold_mask")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by cold/warm: ---

            count  pos_rate
cold_mask                  
False      358494  0.344717
True        14539  0.140312


Cold-start interactions constitute a small but distinct subset (~3.9%) with a substantially lower positive rate, validating the need for specialised cold-start handling within the unified pipeline.

In [46]:
print("--- Counts by scenario: --- \n")
print(pred_df.groupby("scene_idx")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by scenario: --- 

            count  pos_rate
scene_idx                  
0           21682  0.041232
1           91798  0.231726
2          189948  0.467444
3           42514  0.297408
4           27091  0.074527


Interaction volume and outcome vary markedly across scenarios, justifying the use of scenario-aware routing rather than a single shared prediction pathway.

In [51]:
print("--- Counts by alpha_bin: ---\n")
print(pred_df.groupby("alpha_bin")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by alpha_bin: ---

             count  pos_rate
alpha_bin                   
alpha_high  365852  0.340821
alpha_low     4238  0.120340
alpha_mid     2943  0.142372


Most predictions rely heavily on ID-based embeddings (alpha_high), while lower-α cases correspond to colder items where meta-embeddings contribute more strongly.

In [52]:
print("--- Routing summary (expert selection counts): ---\n")
print(pred_df["gate_expert"].value_counts().sort_index())

--- Routing summary (expert selection counts): ---

gate_expert
0    189948
1     91798
2     91287
Name: count, dtype: int64


Multiple experts are actively selected at inference time, indicating that the MMoE router learns differentiated representations rather than collapsing to a single expert.

In [53]:
print("--- Gate max weight summary: ---\n")
print(pred_df["gate_max"].describe())

--- Gate max weight summary: ---

count    373033.000000
mean          0.369072
std           0.105901
min           0.284068
25%           0.316574
50%           0.316574
75%           0.386024
max           0.650946
Name: gate_max, dtype: float64


Gate confidence remains moderate (mean ≈ 0.37), suggesting soft expert mixtures rather than overconfident hard routing, which supports stable generalisation across scenarios.

In [54]:
print("--- Gate entropy summary: ---\n")
print(pred_df["gate_entropy"].describe())

--- Gate entropy summary: ---

count    373033.000000
mean          1.292807
std           0.104611
min           1.026001
25%           1.256635
50%           1.350762
75%           1.350762
max           1.379710
Name: gate_entropy, dtype: float64


High but non-maximal gate entropy indicates controlled uncertainty in expert selection, consistent with smooth scenario-aware routing rather than random or collapsed behaviour.